# 41 검색문 보강 후 전체 검색 비교
운영 코드·OCR·이전 결과는 읽기 전용입니다. 실험은 순차 실행합니다.
이 노트북은 입력 → 실행 상태 → 평가 → 비용을 저장 결과로 확인합니다. 유료 호출은 자동 실행하지 않습니다.
장시간 실행·원응답 journaling·동일 payload 캐시·메모리 중단/재개가 필요해 `41_run_query_expansion.py`를 별도 runner로 사용합니다.
재현: 기존 승인 환경에서 `python -B notebooks/scripts/41_run_query_expansion.py` 실행 후 이 노트북의 전체 표시 셀을 실행합니다.
모델·자료·평가 기준은 protocol.json과 소스 해시로 고정합니다. 같은 질문 재호출 분산은 측정하지 않습니다.


In [1]:
from pathlib import Path
import json
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "notebooks/data/41_query_expansion"
def read(name): return json.loads((DATA / name).read_text())
protocol = read("protocol.json")
print(json.dumps(protocol, ensure_ascii=False, indent=2)[:10000])


{
  "answer_question": "original user question",
  "baseline_source": "40_new_query_comparison",
  "conditions": {
    "answer_output": 2400,
    "candidate": 20,
    "component": 50,
    "parent_bundle": "budget5",
    "top_k": 3
  },
  "dataset_sha256": "efeecd952ddabc103d1831aff91d13f8d2f60563672d6ef5da1bb89c17e6a3fa",
  "expansions": {
    "N01": {
      "added_terms": [],
      "boolean_note": "OR/AND text is retained; vector/BM25 ranking does not enforce Boolean filtering.",
      "fallback_reason": null,
      "original_question": "넷플릭스 구독료를 7% 이상 할인받을 수 있는 카드 후보와 조건을 알려줘",
      "relations": [],
      "search_question": "넷플릭스 구독료를 7% 이상 할인받을 수 있는 카드 후보와 조건을 알려줘\n검색 표현: (넷플릭스 할인)"
    },
    "N02": {
      "added_terms": [],
      "boolean_note": "OR/AND text is retained; vector/BM25 ranking does not enforce Boolean filtering.",
      "fallback_reason": null,
      "original_question": "쿠팡 쇼핑 결제액의 5% 이상을 적립하는 카드를 찾고 싶어",
      "relations": [],
      "search_question": "쿠팡 쇼핑 결제액

## 실행 결과
검색·답변 실패도 분모에 포함합니다. 일반/모호/자료 미확인 질문은 양성 지표에서 분리합니다.


In [2]:
print(json.dumps(read("run_summary.json"), ensure_ascii=False, indent=2))


{
  "answer_rows": 36,
  "answer_success_rows": 36,
  "cost": {
    "billed_cost_usd": null,
    "duplicate_saved_responses": 0,
    "known_cost_usd": "0.020604415",
    "known_priced_records": 21,
    "record_count": 21,
    "tokens": {
      "cache_write_tokens": 84523,
      "cached_input_tokens": 32774,
      "embedding_tokens": 815,
      "input_tokens": 117357,
      "ordinary_input_tokens": 60,
      "output_tokens": 19378
    },
    "unknown_cost_records": 0
  },
  "expected": 36,
  "production_adopted": false,
  "resources": {
    "billed_cost": "미측정",
    "cpu_seconds_this_process": 2214.1671049999995,
    "gpu": "CPU 실행",
    "last_available_kib": 2778588,
    "last_required_kib": 1048576,
    "minimum_available_kib": 2713804,
    "new_bge_pairs_this_process": 404,
    "peak_rss_kib": 2619856,
    "singleton_calls": 404,
    "transmitted_tokens": "미측정",
    "wall_seconds_this_process": 831.7831856419944
  },
  "retrieval_rows": 36,
  "retrieval_success_rows": 36
}


## 평가
Hit@3는 상위3개 정답 카드 포함, MRR은 첫 정답 순위 역수 평균, 답변충족은 원문 상대 자동 조건 검사의 전체 통과입니다. 높을수록 좋습니다.
카드가 검색돼도 혜택·조건을 답변에서 빠뜨리면 답변충족에 실패할 수 있습니다. 문자열·동의어·부정문 오탐과 불완전한 원문 관계를 원응답으로 따로 점검합니다.
benefit과 parent-child는 청킹 외 재정렬 프로필에도 차이가 있습니다.


In [3]:
import pandas as pd
evaluation = read("evaluation_summary.json")
display(pd.DataFrame(evaluation["metrics"]))
print(json.dumps(evaluation["paired"], ensure_ascii=False, indent=2))


,all_n,complete,failures,group,hit,mode,mrr,ownership_failures,positive_n,profile,scope_failures,unsupported_percent_claims
0,18,14,0,all,14,U,0.855556,0,15,card_page_section_benefit,0,1
1,18,15,0,all,15,S_THEN_U,0.966667,0,15,card_page_section_benefit,0,1
2,18,15,0,all,15,S_PLUS_U,0.966667,0,15,card_page_section_benefit,0,1
3,18,13,0,all,13,Q_EXPAND_U,0.833333,0,15,card_page_section_benefit,0,1
4,18,13,0,all,13,U,0.833333,0,15,parent_child_bundle,0,0
5,18,14,0,all,15,S_THEN_U,0.966667,0,15,parent_child_bundle,0,0
6,18,15,0,all,15,S_PLUS_U,0.933333,0,15,parent_child_bundle,0,0
7,18,13,0,all,15,Q_EXPAND_U,0.866667,0,15,parent_child_bundle,0,0
8,6,6,0,numeric,6,U,0.805556,0,6,card_page_section_benefit,0,0
9,6,6,0,numeric,6,S_THEN_U,0.916667,0,6,card_page_section_benefit,0,0


[
  {
    "baseline": "U",
    "candidate": "Q_EXPAND_U",
    "changed": [
      {
        "query_id": "N02",
        "question": "쿠팡 쇼핑 결제액의 5% 이상을 적립하는 카드를 찾고 싶어",
        "result": "loss"
      }
    ],
    "losses": 1,
    "profile": "card_page_section_benefit",
    "ties": 14,
    "wins": 0
  },
  {
    "baseline": "S_THEN_U",
    "candidate": "Q_EXPAND_U",
    "changed": [
      {
        "query_id": "N02",
        "question": "쿠팡 쇼핑 결제액의 5% 이상을 적립하는 카드를 찾고 싶어",
        "result": "loss"
      },
      {
        "query_id": "N15",
        "question": "마트 장보기와 대중교통을 모두 할인받을 수 있는 카드가 필요해",
        "result": "loss"
      }
    ],
    "losses": 2,
    "profile": "card_page_section_benefit",
    "ties": 13,
    "wins": 0
  },
  {
    "baseline": "S_PLUS_U",
    "candidate": "Q_EXPAND_U",
    "changed": [
      {
        "query_id": "N02",
        "question": "쿠팡 쇼핑 결제액의 5% 이상을 적립하는 카드를 찾고 싶어",
        "result": "loss"
      },
      {
        "query_id": "N15",
        "question": "마트 

## 비용·원본 보존
API 실제 물리 요청과 캐시 재사용을 분리합니다. 사용량×기록 단가 환산이며 실제 청구액은 미측정입니다. 시간·CPU·peak RSS는 해당 프로세스의 측정값입니다.


In [4]:
print(json.dumps(read("cost_accounting.json")["summary"], ensure_ascii=False, indent=2))
print(json.dumps(read("preservation.json"), ensure_ascii=False, indent=2))


{
  "billed_cost_usd": null,
  "duplicate_saved_responses": 0,
  "known_cost_usd": "0.020604415",
  "known_priced_records": 21,
  "record_count": 21,
  "tokens": {
    "cache_write_tokens": 84523,
    "cached_input_tokens": 32774,
    "embedding_tokens": 815,
    "input_tokens": 117357,
    "ordinary_input_tokens": 60,
    "output_tokens": 19378
  },
  "unknown_cost_records": 0
}
{
  "changed": [],
  "checked_files": 20066,
  "passed": true
}
